In [6]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
# Import hàm preprocess chuẩn của ResNet50 để dùng Transfer Learning hiệu quả
from tensorflow.keras.applications.resnet50 import preprocess_input 
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, recall_score, f1_score

# --- 1. CONFIGURATION ---
BATCH_SIZE = 64
SPLIT_VAL = 0.2
N_TEST_IMAGES = 3000
EPOCHS = 10 # Tăng lên 10 vì có EarlyStopping lo
IMG_HEIGHT = 224
IMG_WIDTH = 224
CHANNELS = 3

print(f"TensorFlow Version: {tf.__version__}")


TensorFlow Version: 2.19.0


In [7]:

# --- 2. DATASET WRAPPER (Giữ nguyên logic của bạn) ---
class celeb_dataframe():
    '''Wraps the celebA dataset.'''
    def __init__(self, main_folder="/kaggle/input/celeba-dataset", selected_features=[], drop_features=[]):
        self.main_folder = main_folder
        self.images_folder = os.path.join(main_folder, 'img_align_celeba/img_align_celeba')
        self.attr_path = os.path.join(main_folder, "list_attr_celeba.csv")
        self.features_name = []
        self.prepare(drop_features, selected_features)
        
    def prepare(self, drop_features, selected_features):
        if len(selected_features) == 0:
            self.attributes = pd.read_csv(self.attr_path)
            self.num_features = 40
        else:
            self.num_features = len(selected_features)
            selected_features = selected_features.copy()
            selected_features.append("image_id")
            self.attributes = pd.read_csv(self.attr_path)[selected_features]
        
        if len(drop_features) != 0:
            for feature in drop_features:
                if feature in self.attributes:
                    self.attributes = self.attributes.drop(feature, axis=1)
                    self.num_features -= 1
            
        self.attributes.set_index("image_id", inplace=True)
        self.attributes.replace(to_replace=-1, value=0, inplace=True)
        self.attributes["image_id"] = list(self.attributes.index)
        self.features_name = list(self.attributes.columns)[:-1]
        
        return self.attributes

# --- 3. PREPARE DATA & SPLITTING ---
features = [
    'Male', 'Bald', 'Bangs', 'Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Gray_Hair',
    'Straight_Hair', 'Wavy_Hair', 'Receding_Hairline', 'Mustache', 'Sideburns', 
    'No_Beard', 'Eyeglasses', 'Wearing_Hat', 'Wearing_Earrings', 'Wearing_Necklace', 
    'Wearing_Necktie'
]

# Khởi tạo class xử lý dữ liệu
celeb_df = celeb_dataframe()
# Lấy toàn bộ dữ liệu trước
full_celeb_data = celeb_df.prepare(drop_features=[], selected_features=features)

print(f"Tổng số ảnh ban đầu: {full_celeb_data.shape[0]}")
print(f"Số lượng features: {len(celeb_df.features_name)}")

# --- LOGIC TÁCH 3000 ẢNH TEST ---
# 1. Tách 3000 ảnh cuối cùng ra làm tập Test
test_df = full_celeb_data.tail(N_TEST_IMAGES)

# 2. Phần còn lại dùng để Train và Validate
remaining_df = full_celeb_data.iloc[:-N_TEST_IMAGES]

# 3. Chia phần còn lại thành Train và Val theo tỷ lệ SPLIT_VAL
split_ind = int((1 - SPLIT_VAL) * remaining_df.shape[0])
train_df = remaining_df[:split_ind]
validation_df = remaining_df[split_ind:]

print(f"-> Train set: {train_df.shape[0]}")
print(f"-> Val set:   {validation_df.shape[0]}")
print(f"-> Test set:  {test_df.shape[0]}")



Tổng số ảnh ban đầu: 202599
Số lượng features: 18
-> Train set: 159679
-> Val set:   39920
-> Test set:  3000


In [8]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input, # <--- Dùng cái này cho Transfer Learning chuẩn
    rotation_range=20, 
    width_shift_range=0.2, 
    height_shift_range=0.2, 
    shear_range=0.2, 
    zoom_range=0.2, 
    horizontal_flip=True, 
    fill_mode='nearest'
)

# Validation & Test cũng phải dùng preprocess_input
valid_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df, directory=celeb_df.images_folder, x_col='image_id', y_col=celeb_df.features_name,
    target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='raw', shuffle=True
)

validation_generator = valid_test_datagen.flow_from_dataframe(
    dataframe=validation_df, directory=celeb_df.images_folder, x_col='image_id', y_col=celeb_df.features_name,
    target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='raw', shuffle=False
)

test_generator = valid_test_datagen.flow_from_dataframe(
    dataframe=test_df, directory=celeb_df.images_folder, x_col='image_id', y_col=celeb_df.features_name,
    target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='raw', shuffle=False
)

# --- 5. TÍNH TOÁN CLASS WEIGHTS (CHO LOSS) ---
# Tính toán trọng số để xử lý mất cân bằng dữ liệu
pos_counts = train_df[celeb_df.features_name].sum()
total_counts = len(train_df)
# Công thức: Weight = (Total - Pos) / Pos
# Nếu class ít xuất hiện, weight sẽ cao
pos_weights_val = (total_counts - pos_counts) / pos_counts
# Chuyển về tensor float32 để dùng trong hàm loss
pos_weights = tf.constant(pos_weights_val.values, dtype=tf.float32)

print("Positive Weights calculated.")

# --- 6. HÀM LOSS ĐÃ SỬA LỖI ---
def weighted_binary_crossentropy(y_true, y_pred):
    # Ép kiểu dữ liệu để tránh lỗi type mismatch
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    # 1. Tính Binary Crossentropy cho TỪNG PHẦN TỬ (giữ nguyên shape [batch, features])
    # Sử dụng công thức toán học trực tiếp để tránh việc Keras tự reduce_mean
    # Loss = - (y_true * log(y_pred) + (1 - y_true) * log(1 - y_pred))
    # Cộng thêm epsilon để tránh log(0) = -inf
    epsilon = tf.keras.backend.epsilon()
    y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
    
    bce = - (y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
    
    # 2. Tạo ma trận trọng số (shape [batch, features])
    # Nếu mẫu là dương (y_true=1), nhân với pos_weight. Nếu âm, nhân với 1.
    weight_vector = y_true * pos_weights + (1. - y_true)
    
    # 3. Nhân trọng số vào Loss
    weighted_bce = weight_vector * bce
    
    # 4. Tính trung bình tổng loss
    return tf.reduce_mean(weighted_bce)

Found 159679 validated image filenames.
Found 39920 validated image filenames.
Found 3000 validated image filenames.
Positive Weights calculated.


In [9]:
# --- 7. MODEL DEFINITION: RESNET50 (TRANSFER LEARNING) ---
def build_resnet50_classifier(num_features):
    base = ResNet50(
        input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
        weights='imagenet', # <--- Transfer Learning (Dùng weights đã học sẵn)
        include_top=False,
        pooling="avg"
    )

    x = base.output
    x = Dense(1024, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x) # Tăng Dropout lên 0.4 để chống overfitting khi dùng pretrained
    
    output = Dense(num_features, activation='sigmoid')(x)

    model = Model(inputs=base.input, outputs=output)
    return model

model = build_resnet50_classifier(len(celeb_df.features_name))

model.compile(
    loss=weighted_binary_crossentropy, # <--- Dùng Loss đã sửa
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), # Giảm LR khi dùng pretrained weights
    metrics=['binary_accuracy']
)

# --- 8. TRAINING ---
os.makedirs("/kaggle/working/ckpts/", exist_ok=True)
earlystop = EarlyStopping(monitor="val_binary_accuracy", patience=4, restore_best_weights=True)

# Sửa lại định dạng tên file checkpoint để tránh lỗi format
checkpoint_filepath = "/kaggle/working/ckpts/weights-ResNet50.weights.h5"
model_checkpoint = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True, monitor='val_binary_accuracy', mode='max', save_best_only=True, verbose=1
)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:

print("Starting Training with Weighted Loss & ImageNet Weights...")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    steps_per_epoch=len(train_generator),
    validation_data=validation_generator,
    validation_steps=len(validation_generator),
    callbacks=[earlystop, model_checkpoint],
    verbose=1
)



Starting Training with Weighted Loss & ImageNet Weights...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1767498369.955373     122 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1767498405.560583     122 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2495/2495 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - binary_accuracy: 0.7794 - loss: 0.6724
Epoch 1: val_binary_accuracy improved from -inf to 0.88403, saving model to /kaggle/working/ckpts/weights-ResNet50.weights.h5
2495/2495 ━━━━━━━━━━━━━━━━━━━━ 3258s 1s/step - binary_accuracy: 0.7794 - loss: 0.6723 - val_binary_accuracy: 0.8840 - val_loss: 0.4009
Epoch 2/10
2495/2495 ━━━━━━━━━━━━━━━━━━━━ 0s 826ms/step - binary_accuracy: 0.8900 - loss: 0.4012
Epoch 2: val_binary_accuracy improved from 0.88403 to 0.89309, saving model to /kaggle/working/ckpts/weights-ResNet50.weights.h5
2495/2495 ━━━━━━━━━━━━━━━━━━━━ 2221s 890ms/step - binary_accuracy: 0.8900 - loss: 0.4012 - val_binary_accuracy: 0.8931 - val_loss: 0.3691
Epoch 3/10
2495/2495 ━━━━━━━━━━━━━━━━━━━━ 0s 832ms/step - binary_accuracy: 0.8944 - loss: 0.3732
Epoch 3: val_binary_accuracy improved from 0.89309 to 0.89820, saving model to /kaggle/working/ckpts/weights-ResNet50.weights.h5
2495/2495 ━━━━━━━━━━━━━━━━━━━━ 2243s 899ms/step - binary_accuracy:

In [12]:
# --- 9. EVALUATION ---
print("\n" + "="*40)
print(f"ĐANG ĐÁNH GIÁ TRÊN {N_TEST_IMAGES} ẢNH TEST...")
print("="*40)

test_generator.reset()
preds_prob = model.predict(test_generator, verbose=1)

# Thử nghiệm với các ngưỡng khác nhau nếu cần (ví dụ 0.4 thay vì 0.5)
threshold = 0.5 
preds_bin = (preds_prob > threshold).astype(int)
y_true = test_df[celeb_df.features_name].values

# Metrics
acc = accuracy_score(y_true.flatten(), preds_bin.flatten()) # Tính accuracy phẳng cho chính xác hơn
macro_recall = recall_score(y_true, preds_bin, average='macro', zero_division=0)
macro_f1 = f1_score(y_true, preds_bin, average='macro', zero_division=0)

print("\nKẾT QUẢ ĐÁNH GIÁ (RESNET50 + IMAGENET + WEIGHTED LOSS):")
print("-" * 30)
print(f"Flat Accuracy:    {acc:.4f}")
print(f"Macro Recall:     {macro_recall:.4f}")
print(f"Macro F1-Score:   {macro_f1:.4f}")
print("-" * 30)

# Save Final Model
model.save("/kaggle/working/Final_ResNet50_Transfer.keras")


ĐANG ĐÁNH GIÁ TRÊN 3000 ẢNH TEST...
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 204ms/step

KẾT QUẢ ĐÁNH GIÁ (RESNET50 + IMAGENET + WEIGHTED LOSS):
------------------------------
Flat Accuracy:    0.9076
Macro Recall:     0.9320
Macro F1-Score:   0.7300
------------------------------
